# RecTools Transformer Models Tutorial：SASRec / BERT4Rec PyTorch 中文版

这份 notebook 对应 RecTools 官方的 **RecSys Transformer Models Tutorial**，但这里不依赖 RecTools 包本身，而是用纯 PyTorch 把核心流程拆开实现，方便你学习底层原理。

我们会覆盖：

- SASRec / BERT4Rec 的训练目标差异。
- sequential recommendation 的数据组织方式。
- item sequence padding、causal mask、BERT mask。
- next-item prediction 和 masked-item prediction。
- softmax、sampled BCE、BPR 三种 loss。
- Top-K 推荐评估流程。
- item features、交叉验证、item-to-item 推荐、冷启动推理。


## 1. 导入依赖

这里完全使用 PyTorch。`src/transformer_seqrec.py` 中放了模型、Dataset、loss 和评估函数，notebook 负责解释和串流程。

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

PROJECT_DIR = Path.cwd().parent
SRC_DIR = PROJECT_DIR / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

from transformer_seqrec import (
    BERT4RecDataset,
    IGNORE_INDEX,
    MASK_TOKEN,
    PAD_TOKEN,
    SASRecDataset,
    SequentialTransformerRec,
    bpr_loss,
    build_encoding,
    encode_user_sequences,
    full_softmax_loss,
    item_to_item_recommendations,
    leave_one_out_split,
    make_toy_interactions,
    ranking_metrics,
    recommend_topk,
    sampled_bce_loss,
    train_one_model,
)

seed = 42
np.random.seed(seed)
torch.manual_seed(seed)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"当前训练设备：{device}")


## 2. 准备顺序推荐数据

序列推荐和 CTR 表格特征不一样：核心输入不是一行稀疏特征，而是用户按时间排序后的历史 item 序列。这里用一个小型 toy 数据集离线演示，真实项目里只要把 `interactions` 换成真实 `(user_id, item_id, timestamp)` 表即可。

In [ ]:
interactions, item_features = make_toy_interactions()
display(interactions.head(10))
display(item_features.head(10))

print("用户数：", interactions["user_id"].nunique())
print("物品数：", interactions["item_id"].nunique())
print("交互数：", len(interactions))


## 3. item 编码、padding 和序列截断

模型只认识整数 token。我们保留两个特殊 token：

- `0 = PAD`：补齐序列长度。
- `1 = MASK`：BERT4Rec 训练时遮住 item。

真实 RecTools tutorial 里也会先把交互表转成用户 session。这里同样按用户和时间排序，并只保留最近 `max_len` 个行为。

In [ ]:
max_len = 6
encoding = build_encoding(item_features, max_len=max_len)
user_sequences = encode_user_sequences(interactions, encoding)

for user_id, seq in list(user_sequences.items())[:3]:
    readable = [encoding.token_to_item[token] for token in seq]
    print(user_id, "原始序列：", readable)

print("词表大小（含 PAD/MASK）：", encoding.vocab_size)
print("类别特征数（含 PAD/MASK）：", len(encoding.category_to_id))


## 4. SASRec 数据组织：shifted sequence

SASRec 的目标是 **next-item prediction**。训练时输入是历史序列，标签是右移一位后的下一个 item。由于它不能偷看未来，所以 Transformer 里会使用 causal attention mask。

In [ ]:
sasrec_dataset = SASRecDataset(user_sequences, max_len=max_len)
sasrec_loader = DataLoader(sasrec_dataset, batch_size=4, shuffle=True)

example = sasrec_dataset[0]
print("用户：", example["user_id"])
print("输入 token：", example["input_ids"].tolist())
print("标签 token：", example["labels"].tolist())
print("输入 item：", [encoding.token_to_item.get(token, "<IGNORE>") for token in example["input_ids"].tolist()])
print("标签 item：", [encoding.token_to_item.get(token, "<IGNORE>") if token != IGNORE_INDEX else "<IGNORE>" for token in example["labels"].tolist()])


## 5. BERT4Rec 数据组织：masked item prediction

BERT4Rec 不是只看左侧历史，而是在整条序列中随机遮住一部分 item，然后预测被遮住的 item。它使用双向 attention，因此可以同时利用 mask 左右两侧的信息。

In [ ]:
bert_dataset = BERT4RecDataset(user_sequences, max_len=max_len, mask_prob=0.4, seed=seed)
bert_loader = DataLoader(bert_dataset, batch_size=4, shuffle=True)

example = bert_dataset[0]
print("用户：", example["user_id"])
print("输入 token：", example["input_ids"].tolist())
print("标签 token：", example["labels"].tolist())
print("输入 item：", [encoding.token_to_item.get(token, "<IGNORE>") for token in example["input_ids"].tolist()])
print("标签 item：", [encoding.token_to_item.get(token, "<IGNORE>") if token != IGNORE_INDEX else "<IGNORE>" for token in example["labels"].tolist()])


## 6. 定义 SASRec 和 BERT4Rec

两个模型共用同一个 `SequentialTransformerRec`：

- `causal=True` 时就是 SASRec，只允许当前位置看过去。
- `causal=False` 时就是 BERT4Rec，可以看完整序列。

`use_item_features=True` 表示把 item 类别 embedding 加到 item id embedding 上，这对应 RecTools tutorial 中的 item features 思路。

In [ ]:
def make_model(causal: bool, use_item_features: bool = True):
    return SequentialTransformerRec(
        vocab_size=encoding.vocab_size,
        max_len=max_len,
        token_to_category=encoding.token_to_category,
        category_vocab_size=len(encoding.category_to_id),
        hidden_dim=48,
        num_heads=4,
        num_layers=2,
        dropout=0.1,
        use_item_features=use_item_features,
        causal=causal,
    ).to(device)

sasrec = make_model(causal=True, use_item_features=True)
bert4rec = make_model(causal=False, use_item_features=True)

print("SASRec 参数量：", sum(p.numel() for p in sasrec.parameters()))
print("BERT4Rec 参数量：", sum(p.numel() for p in bert4rec.parameters()))


## 7. 多种 loss 选择

RecTools tutorial 强调 Transformer 模型可以选择不同 loss。这里演示三类：

- `softmax`：对全量 item 分类，最直观。
- `sampled_bce`：只采样一小部分负样本，适合物品数很大时降成本。
- `bpr`：pairwise 排序目标，让正样本分数高于负样本。

In [ ]:
batch = next(iter(sasrec_loader))
input_ids = batch["input_ids"].to(device)
labels = batch["labels"].to(device)
logits = sasrec(input_ids)

print("softmax loss：", round(full_softmax_loss(logits, labels).item(), 4))
print("sampled BCE loss：", round(sampled_bce_loss(logits, labels, sasrec.vocab_size, num_negatives=4).item(), 4))
print("BPR loss：", round(bpr_loss(logits, labels, sasrec.vocab_size, num_negatives=4).item(), 4))


## 8. 训练 SASRec：next-item prediction

下面用完整 softmax 训练 SASRec。toy 数据很小，所以我们只训练十几轮，目的是验证流程，而不是追求真实线上效果。

In [ ]:
sasrec_optimizer = torch.optim.Adam(sasrec.parameters(), lr=1e-3, weight_decay=1e-5)
sasrec_losses = train_one_model(
    sasrec,
    sasrec_loader,
    sasrec_optimizer,
    loss_name="softmax",
    device=device,
    epochs=14,
)
print("SASRec 每轮 loss：", [round(value, 4) for value in sasrec_losses])


## 9. 训练 BERT4Rec：masked item prediction

BERT4Rec 使用 mask token 预测被遮住的 item。注意它的目标不是每个位置都预测，而只在标签不等于 `IGNORE_INDEX` 的位置计算 loss。

In [ ]:
bert_optimizer = torch.optim.Adam(bert4rec.parameters(), lr=1e-3, weight_decay=1e-5)
bert_losses = train_one_model(
    bert4rec,
    bert_loader,
    bert_optimizer,
    loss_name="softmax",
    device=device,
    epochs=14,
)
print("BERT4Rec 每轮 loss：", [round(value, 4) for value in bert_losses])


## 10. Top-K 推荐评估

常见流程是 leave-one-out：每个用户最后一个 item 作为测试目标，前面的序列作为历史。然后看模型推荐的 Top-K 里是否命中该目标，并计算 HitRate@K、NDCG@K、MRR@K。

In [ ]:
train_sequences, test_targets = leave_one_out_split(user_sequences)

sasrec_metrics = ranking_metrics(
    sasrec, train_sequences, test_targets, max_len=max_len, k=5, device=device, bert_style=False
)
bert_metrics = ranking_metrics(
    bert4rec, train_sequences, test_targets, max_len=max_len, k=5, device=device, bert_style=True
)

print("SASRec Top-K 指标：", sasrec_metrics)
print("BERT4Rec Top-K 指标：", bert_metrics)


## 11. item features 的作用

这里的 item feature 是类别特征。模型把 `item id embedding + category embedding + position embedding` 加在一起送入 Transformer。你可以把 `use_item_features=False` 关掉，对比只用 item id 的版本。

In [ ]:
sasrec_no_features = make_model(causal=True, use_item_features=False)
optimizer_no_features = torch.optim.Adam(sasrec_no_features.parameters(), lr=1e-3, weight_decay=1e-5)
losses_no_features = train_one_model(
    sasrec_no_features,
    sasrec_loader,
    optimizer_no_features,
    loss_name="softmax",
    device=device,
    epochs=6,
)
metrics_no_features = ranking_metrics(
    sasrec_no_features, train_sequences, test_targets, max_len=max_len, k=5, device=device
)
print("不用 item features 的 SASRec loss：", [round(value, 4) for value in losses_no_features])
print("不用 item features 的 SASRec 指标：", metrics_no_features)


## 12. 简单交叉验证

真实项目中，序列推荐通常按时间切分验证集。这里为了演示交叉验证的写法，用用户维度做两个 fold：每次训练一部分用户，再在另一部分用户的 leave-one-out 目标上评估。

In [ ]:
user_ids = sorted(user_sequences)
folds = [user_ids[::2], user_ids[1::2]]
cv_rows = []

for fold_id, valid_users in enumerate(folds, start=1):
    train_users = [user for user in user_ids if user not in valid_users]
    fold_train_sequences = {user: user_sequences[user] for user in train_users}
    fold_valid_sequences = {user: user_sequences[user] for user in valid_users}

    fold_dataset = SASRecDataset(fold_train_sequences, max_len=max_len)
    fold_loader = DataLoader(fold_dataset, batch_size=4, shuffle=True)
    fold_model = make_model(causal=True, use_item_features=True)
    fold_optimizer = torch.optim.Adam(fold_model.parameters(), lr=1e-3)
    train_one_model(fold_model, fold_loader, fold_optimizer, loss_name="softmax", device=device, epochs=5)

    fold_history, fold_targets = leave_one_out_split(fold_valid_sequences)
    fold_metrics = ranking_metrics(fold_model, fold_history, fold_targets, max_len=max_len, k=5, device=device)
    fold_metrics["fold"] = fold_id
    cv_rows.append(fold_metrics)

pd.DataFrame(cv_rows)


## 13. item-to-item 推荐

Transformer 序列模型训练完后，item embedding 也学到了相似性。可以用余弦相似度做 item-to-item 推荐，比如给一个物品找相似物品。

In [ ]:
anchor_item = "I5"
anchor_token = encoding.item_to_token[anchor_item]
item_to_item_recommendations(sasrec, anchor_token, encoding.token_to_item, k=5)


## 14. 冷启动用户推理

如果一个新用户只有很短的历史，仍然可以左侧 padding 后进行推理。只要历史里的 item 在训练词表里，模型就能给出 Top-K 推荐。

In [ ]:
cold_history_items = ["I1", "I5"]
cold_history_tokens = [encoding.item_to_token[item_id] for item_id in cold_history_items]
recommend_tokens = recommend_topk(
    sasrec,
    cold_history_tokens,
    max_len=max_len,
    k=5,
    device=device,
    seen_items=set(cold_history_tokens),
)
recommend_items = [encoding.token_to_item[token] for token in recommend_tokens]
print("冷启动用户历史：", cold_history_items)
print("推荐结果：", recommend_items)


## 15. 小结

- SASRec 更像自回归序列模型：只看过去，预测下一个 item。
- BERT4Rec 更像 masked language model：随机遮住 item，用双向上下文恢复它。
- padding 和 mask 是序列推荐能否正确训练的关键。
- loss 可以根据物品规模和目标选择：全量 softmax 简单，sampled BCE 和 BPR 更适合大规模候选。
- Top-K 评估重点看目标 item 是否被排到前面，常用 HitRate、NDCG、MRR。

这份 notebook 是学习版，真实数据上可以替换 `make_toy_interactions()`，保留后面的模型、训练和评估流程。